# Central Limit Theorem (CLT) Simulation
**CLT:** For a population with mean $\mu$ and finite variance $\sigma^2$, the distribution of the sample mean $\bar{X}$ from samples of size $n$ approaches a Normal distribution as $n$ grows:

$$\bar{X} \sim N\left(\mu, \frac{\sigma^2}{n}\right), \quad SE = \frac{\sigma}{\sqrt{n}}$$

This notebook asks for any dataset, takes a numeric column as the "population", and simulates CLT. Part B uses built-in distributions (no dataset needed).

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
np.random.seed(42)

# Part A: CLT on Your Dataset

## 2. Select Dataset from File Explorer

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
file_path = filedialog.askopenfilename(
    title="Select your dataset",
    filetypes=[("Data files", "*.csv *.xlsx *.xls *.txt *.tsv"), ("All files", "*.*")]
)
root.destroy()

if not file_path:
    raise SystemExit("No file selected. Re-run this cell.")

print("Selected file:", file_path)

if file_path.endswith('.csv'):
    df = pd.read_csv(file_path)
elif file_path.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_path)
else:
    df = pd.read_csv(file_path, sep=None, engine='python')

print("Shape:", df.shape)
df.head()

## 3. Choose a Numeric Column (the Population)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:")
for i, c in enumerate(num_cols):
    print(f"{i}: {c}")

col = input("\nEnter column name (or index number): ").strip()
if col.isdigit() and col not in df.columns:
    col = num_cols[int(col)]

population = df[col].dropna().values
N = len(population)
mu = population.mean()
sigma = population.std(ddof=0)

print(f"\nColumn: {col}")
print(f"Population size N = {N}")
print(f"Population mean (mu)    = {mu:.4f}")
print(f"Population std (sigma)  = {sigma:.4f}")
print(f"Skewness = {stats.skew(population):.4f}, Kurtosis = {stats.kurtosis(population):.4f}")

## 4. Population Distribution (usually NOT normal)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(population, kde=True, ax=ax[0], color='steelblue')
ax[0].axvline(mu, color='r', ls='--', label='mean'); ax[0].legend()
ax[0].set_title(f'Population: {col}')

sns.boxplot(x=population, ax=ax[1], color='lightblue')
ax[1].set_title('Boxplot')

stats.probplot(population, dist='norm', plot=ax[2])
ax[2].set_title('Q-Q plot of population')
plt.tight_layout(); plt.show()

# Normality test (on a subsample if large, Shapiro limit ~5000)
sub = np.random.choice(population, size=min(5000, N), replace=False)
stat, p = stats.shapiro(sub)
print(f"Shapiro-Wilk on population: stat={stat:.4f}, p-value={p:.4g}")
print("Population is", "NORMAL" if p > 0.05 else "NOT normal", "(alpha = 0.05)")

## 5. CLT Simulation Function
Draw `n_sims` random samples of size `n` and record each sample mean.

In [ ]:
def simulate_sample_means(data, n, n_sims=5000, replace=True):
    # replace=True treats the data as an infinite population (standard CLT simulation)
    return np.array([np.random.choice(data, size=n, replace=replace).mean()
                     for _ in range(n_sims)])

## 6. CLT in Action: Sample Means for Different Sample Sizes

In [ ]:
sample_sizes = [1, 2, 5, 10, 30, 50, 100]   # change as you like
n_sims = 5000

sample_means = {n: simulate_sample_means(population, n, n_sims) for n in sample_sizes}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()
for ax, n in zip(axes, sample_sizes):
    m = sample_means[n]
    sns.histplot(m, kde=False, stat='density', bins=40, ax=ax, color='skyblue')
    x = np.linspace(m.min(), m.max(), 300)
    ax.plot(x, stats.norm.pdf(x, mu, sigma/np.sqrt(n)), 'r-', lw=2, label='Theoretical N(mu, sigma/sqrt(n))')
    ax.set_title(f'n = {n}')
    ax.legend(fontsize=7)
for ax in axes[len(sample_sizes):]:
    ax.axis('off')
plt.suptitle(f'Distribution of Sample Means - {col}', fontsize=14)
plt.tight_layout(); plt.show()

## 7. Theoretical vs Empirical Mean and Standard Error

In [ ]:
rows = []
for n in sample_sizes:
    m = sample_means[n]
    sub = m[:5000]
    sh_p = stats.shapiro(sub)[1]
    rows.append({
        'n': n,
        'Mean of means': m.mean(),
        'Population mu': mu,
        'Empirical SE': m.std(ddof=1),
        'Theoretical SE (sigma/sqrt(n))': sigma/np.sqrt(n),
        'Skewness': stats.skew(m),
        'Excess Kurtosis': stats.kurtosis(m),
        'Shapiro p-value': sh_p,
        'Normal? (p>0.05)': sh_p > 0.05
    })
summary = pd.DataFrame(rows).set_index('n')
summary.round(4)

## 8. Q-Q Plots of Sample Means (points on the line = normal)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()
for ax, n in zip(axes, sample_sizes):
    stats.probplot(sample_means[n], dist='norm', plot=ax)
    ax.set_title(f'Q-Q plot, n = {n}')
for ax in axes[len(sample_sizes):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 9. Standard Error Shrinks as 1/sqrt(n)

In [ ]:
emp_se = [sample_means[n].std(ddof=1) for n in sample_sizes]
theo_se = [sigma/np.sqrt(n) for n in sample_sizes]

plt.figure(figsize=(7, 5))
plt.plot(sample_sizes, emp_se, 'o-', label='Empirical SE')
plt.plot(sample_sizes, theo_se, 's--', label='Theoretical SE = sigma/sqrt(n)')
plt.xlabel('Sample size n'); plt.ylabel('Standard error')
plt.title('Standard Error vs Sample Size'); plt.legend(); plt.grid(alpha=.3)
plt.show()

## 10. Overlay of All Sample Sizes

In [ ]:
plt.figure(figsize=(10, 5))
for n in [1, 5, 30, 100]:
    sns.kdeplot(sample_means[n], label=f'n={n}', fill=True, alpha=.25)
plt.axvline(mu, color='k', ls='--', label='mu')
plt.title('Sampling distribution narrows and becomes normal as n increases')
plt.legend(); plt.show()

## 11. Confidence Interval Coverage (application of CLT)
If CLT holds, about 95% of the 95% CIs built from samples should contain the true mean.

In [ ]:
n = 50
trials = 2000
covered = 0
for _ in range(trials):
    s = np.random.choice(population, size=n, replace=True)
    se = s.std(ddof=1) / np.sqrt(n)
    lo, hi = s.mean() - 1.96*se, s.mean() + 1.96*se
    covered += (lo <= mu <= hi)

print(f"Sample size n = {n}, trials = {trials}")
print(f"Coverage of 95% CI: {covered/trials*100:.2f}%  (expected ~95%)")

## 12. Interactive Check: Try Your Own n

In [ ]:
n_user = int(input("Enter a sample size n to simulate: "))
sims_user = int(input("Enter number of simulations (e.g. 5000): "))

m = simulate_sample_means(population, n_user, sims_user)
print(f"Mean of sample means : {m.mean():.4f}   (mu = {mu:.4f})")
print(f"Empirical SE         : {m.std(ddof=1):.4f}   (theory = {sigma/np.sqrt(n_user):.4f})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(m, stat='density', bins=40, ax=ax[0], color='skyblue')
x = np.linspace(m.min(), m.max(), 300)
ax[0].plot(x, stats.norm.pdf(x, mu, sigma/np.sqrt(n_user)), 'r-', lw=2)
ax[0].set_title(f'Sample means, n={n_user}')
stats.probplot(m, dist='norm', plot=ax[1])
plt.tight_layout(); plt.show()

# Part B: CLT with Standard Distributions (no dataset needed)

In [ ]:
distributions = {
    'Uniform(0,1)':        lambda size: np.random.uniform(0, 1, size),
    'Exponential(1)':      lambda size: np.random.exponential(1, size),
    'Poisson(3)':          lambda size: np.random.poisson(3, size),
    'Binomial(10,0.2)':    lambda size: np.random.binomial(10, 0.2, size),
    'Chi-square(2)':       lambda size: np.random.chisquare(2, size),
    'Bimodal (mixture)':   lambda size: np.where(np.random.rand(size) < .5,
                                                 np.random.normal(-3, 1, size),
                                                 np.random.normal(3, 1, size)),
}
ns = [1, 5, 30, 100]
n_sims = 5000

fig, axes = plt.subplots(len(distributions), len(ns), figsize=(16, 3*len(distributions)))
for i, (name, gen) in enumerate(distributions.items()):
    big = gen(200000)                      # approximate the population
    mu_d, sd_d = big.mean(), big.std()
    for j, n in enumerate(ns):
        means = np.array([gen(n).mean() for _ in range(n_sims)])
        ax = axes[i, j]
        ax.hist(means, bins=40, density=True, color='skyblue', edgecolor='white')
        x = np.linspace(means.min(), means.max(), 300)
        ax.plot(x, stats.norm.pdf(x, mu_d, sd_d/np.sqrt(n)), 'r-', lw=1.8)
        ax.set_title(f'{name}, n={n}', fontsize=9)
plt.suptitle('CLT for different parent distributions (red = theoretical normal)', y=1.002, fontsize=14)
plt.tight_layout(); plt.show()

## Key Points (for viva)
- **Statement:** sample means of i.i.d. observations with finite mean and variance are approximately normal for large $n$, whatever the population shape.
- **Mean of sampling distribution** $= \mu$ (unbiased). **Standard error** $= \sigma/\sqrt{n}$.
- **Rule of thumb:** $n \ge 30$ is usually enough; heavily skewed populations need larger $n$.
- **Needs:** independent samples, finite variance (fails for Cauchy distribution).
- **Why it matters in ML/statistics:** justifies confidence intervals, z/t tests, and normal approximations of estimators.
- **Larger n** means a narrower and more normal sampling distribution.